# In this notebook, I am focusing on mastering PyTorch, a dynamic computational graph framework that offers flexibility and control for deep learning research and development. While I am already fluent in static graph frameworks like Keras and TensorFlow, which emphasize simplicity and ease of use, I am now exploring PyTorch to deepen my understanding of dynamic computation and to leverage its strengths for more complex and customizable machine learning tasks.


In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor



Download the trainsing and test samples of the faschion dataset leveraging the `Dataset` and `DataLoader` utils

In [2]:
train_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)


100.0%
100.0%
100.0%
100.0%


In [13]:
batche_size = 64

train_dataloader = DataLoader(train_data, batch_size=batche_size)
test_dataloader = DataLoader(test_data, batch_size=batche_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break


Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


# DataLoader Sampler vs Batch Sampler

## Key Concepts for Interview:

### 1. **Sampler** - Controls individual sample selection
- `RandomSampler`: Random selection (shuffle=True)
- `SequentialSampler`: Sequential selection (shuffle=False) 
- `SubsetRandomSampler`: Random from subset
- `WeightedRandomSampler`: Weighted selection for imbalanced data

### 2. **Batch Sampler** - Controls batch formation
- Groups individual samples into batches
- Has higher precedence than sampler + batch_size
- Useful for custom batching strategies

## Examples with FashionMNIST Dataset


In [9]:
from torch.utils.data import RandomSampler, SequentialSampler, SubsetRandomSampler, WeightedRandomSampler, BatchSampler
import numpy as np

# 1. SAMPLER EXAMPLES

print("=== SAMPLER EXAMPLES ===")

# Default behavior (equivalent to RandomSampler when shuffle=True)
default_loader = DataLoader(train_data, batch_size=4, shuffle=True)
print("Default (shuffle=True): Uses RandomSampler internally")

# Explicit RandomSampler
random_sampler = RandomSampler(train_data)
random_loader = DataLoader(train_data, batch_size=4, sampler=random_sampler)
print("RandomSampler: Randomly selects samples")

# SequentialSampler
sequential_sampler = SequentialSampler(train_data)
sequential_loader = DataLoader(train_data, batch_size=4, sampler=sequential_sampler)
print("SequentialSampler: Selects samples in order (0, 1, 2, ...)")

# SubsetRandomSampler - useful for train/validation splits
subset_indices = list(range(0, 1000))  # First 1000 samples
subset_sampler = SubsetRandomSampler(subset_indices)
subset_loader = DataLoader(train_data, batch_size=4, sampler=subset_sampler)
print("SubsetRandomSampler: Randomly samples from specified indices")


=== SAMPLER EXAMPLES ===
Default (shuffle=True): Uses RandomSampler internally
RandomSampler: Randomly selects samples
SequentialSampler: Selects samples in order (0, 1, 2, ...)
SubsetRandomSampler: Randomly samples from specified indices


In [10]:
# WeightedRandomSampler - for imbalanced datasets
# Let's create weights for demonstration (normally you'd calculate based on class distribution)
num_samples = len(train_data)
weights = torch.ones(num_samples)  # Equal weights for demo
weighted_sampler = WeightedRandomSampler(weights, num_samples=1000, replacement=True)
weighted_loader = DataLoader(train_data, batch_size=4, sampler=weighted_sampler)
print("WeightedRandomSampler: Samples based on weights (useful for imbalanced data)")

print("\n=== BATCH SAMPLER EXAMPLES ===")

# 2. BATCH SAMPLER EXAMPLES

# Custom BatchSampler using RandomSampler
base_sampler = RandomSampler(train_data)
batch_sampler = BatchSampler(base_sampler, batch_size=8, drop_last=True)
batch_sampler_loader = DataLoader(train_data, batch_sampler=batch_sampler)
print("BatchSampler: Groups samples from base sampler into batches")

# Note: When using batch_sampler, you cannot use batch_size, shuffle, sampler, or drop_last
# batch_sampler has higher precedence


WeightedRandomSampler: Samples based on weights (useful for imbalanced data)

=== BATCH SAMPLER EXAMPLES ===
BatchSampler: Groups samples from base sampler into batches


In [11]:
# Let's demonstrate the difference by checking first batch indices
print("\n=== DEMONSTRATING THE DIFFERENCE ===")

# Create a small dataset for demonstration
small_indices = list(range(20))  # Indices 0-19

# Using sampler only
sampler_only = SubsetRandomSampler(small_indices)
loader_sampler = DataLoader(train_data, batch_size=5, sampler=sampler_only)

# Using batch_sampler
base_sampler = SubsetRandomSampler(small_indices)
batch_sampler_demo = BatchSampler(base_sampler, batch_size=5, drop_last=False)
loader_batch_sampler = DataLoader(train_data, batch_sampler=batch_sampler_demo)

print("Sampler approach: DataLoader uses sampler + batch_size internally")
print("Batch sampler approach: You control the entire batching process")



=== DEMONSTRATING THE DIFFERENCE ===
Sampler approach: DataLoader uses sampler + batch_size internally
Batch sampler approach: You control the entire batching process
